In [31]:
import SimplicialComplex as sc
import linear_alg_method as lam
import numpy as np
import json
import scipy.sparse as sp

def read_file(filename):
    with open(filename, 'rb') as f:
        data = f.readlines()
        data = [x.strip() for x in data]
    return data

In [32]:
def gf2_rank_from_dict(d):
    """
    Compute GF(2) rank from sparse row representation.
    d: dict-like mapping row_id -> iterable of column indices where entry is 1.
       Only the row values are used; row_id keys can be anything.
    Returns: rank (int) and optionally the basis dict (pivot -> set of cols)
    """
    basis = {}  # pivot_col -> set of ints (columns with 1)
    for row_id, cols in d.items():
        # convert to a mutable set of column indices (skip empty rows)
        row = set(cols)
        if not row:
            continue
        # reduce row using existing basis
        # Choose pivot convention: use max index (arbitrary but deterministic)
        while row:
            pivot = max(row)
            if pivot in basis:
                # row = row XOR basis[pivot]  (symmetric difference)
                # symmetric_difference_update is efficient in-place
                row.symmetric_difference_update(basis[pivot])
            else:
                # new independent row -> store it in basis
                # store a copy; keep a set for further reductions
                basis[pivot] = set(row)
                break
    rank = len(basis)
    return rank, basis

def gf2_rank_from_dict_intbit(d):
    """
    Use Python int as bitset. Column j corresponds to bit (1 << j).
    d: mapping row_id -> iterable of column indices (non-negative ints).
    Returns rank and basis dict (pivot -> int bitmask)
    """
    basis = {}  # pivot_col -> int (bitmask)
    for row_id, cols in d.items():
        # build bitmask
        row = 0
        for j in cols:
            row ^= (1 << j)
        if row == 0:
            continue
        while row:
            # find pivot as index of highest set bit
            pivot = row.bit_length() - 1  # highest 1-bit index
            if pivot in basis:
                row ^= basis[pivot]
            else:
                basis[pivot] = row
                break
    rank = len(basis)
    return rank, basis


In [37]:
list_facets = [json.loads(facets_bytes) for facets_bytes in read_file("./Julia/bin_mat_31_27")]
facets_bin = [sc.face_to_binary(facet,31) for facet in list_facets[0]]
print(len(facets_bin))
ridge_facet_dict={}
for i in sc.list_2_pow[:31]:
    for facet in facets_bin:
        if facet | i != facet:
            continue
        ridge = facet & (facet ^ i)
        if ridge in ridge_facet_dict:
            ridge_facet_dict[ridge].append(facet)
        else:
            ridge_facet_dict[ridge] = [facet]
rank, basis= gf2_rank_from_dict(ridge_facet_dict)
print("dim of the mod 2 kernel=",len(facets_bin)-rank)
# sp_mat = sp.lil_matrix((len(ridge_facet_dict), len(facets_bin)), dtype=int)
# ridge_list = list(ridge_facet_dict.keys())
# ridge_index = {ridge: i for i, ridge in enumerate(ridge_list)}
# facet_index = {facet: i for i, facet in enumerate(facets_bin)}
# for ridge, facets in ridge_facet_dict.items():
#     for facet in facets:
#         sp_mat[ridge_index[ridge], facet_index[facet]] = 1
# M = sp_mat.toarray().astype(int)
# M = Gauss(M)
# r = rank(M)
# print("rank of the incidence matrix is ", r)



83328
dim of the mod 2 kernel= 1024
